# TD — Classification automatique du genre du roman à partir des personnages

## Contexte

Vous disposez de 20 fichiers issus de l’analyse automatique de romans français à l’aide de **BookNLP**.

* **fichiers `.book`** : contiennent des informations détaillées sur les **personnages** détectés (leur genre, leur nombre de mentions, et les mots qui leur sont associés : *agent*, *patient*, *mod*).
Pour simplifier, vous travaillerez sur les **10 personnages les plus importants** de chaque roman.

Votre objectif sera d’entraîner, à partir d'une **représentation vectorielle (Bag of Words)** des personnages,  un **modèle de classification supervisée** (SVM) capable de prédire le **genre littéraire du roman** auquel appartient chaque personnage (*policier, sentimental*).

---

## Objectifs

L’exercice se décompose en plusieurs étapes : # 1, 2 et 3 sont fournis !

### 1. Lecture et préparation des données

* Charger les fichiers `.book` et extraire, pour chaque personnage, les mots qui lui sont associés (`agent`, `patient`, `mod`).
* Limiter le nombre de personnages à **10 par roman**.
* Charger le fichier `genre_labels.json`, qui relie chaque roman (`.txt`) à son **genre littéraire**.

### 2. Construction du vocabulaire

* À partir de l’ensemble des personnages, construire la liste des **1000 mots les plus fréquents** (*Most Frequent Words*).
* Cette liste constituera votre **vocabulaire commun**.

### 3. Construction des représentations vectorielles

* Pour chaque personnage, calculer un **Bag of Words relatif** (fréquences normalisées des 1000 mots les plus fréquents).
* Organiser ces données dans un **DataFrame** :

  * Chaque ligne correspond à un **personnage**.
  * Chaque colonne à un **mot du vocabulaire**.
  * Ajouter une colonne `genre` indiquant le **genre du roman** d’origine.

### 4. Baseline et séparation du jeu de données

* Construire un **modèle de base** prédisant la **classe majoritaire**.
* Séparer les données en **jeu d’entraînement (80 %)** et **jeu de test (20 %)**, en préservant la distribution des genres (`stratify=y`).

### 5. Entraînement d’un modèle SVM

* Entraîner un **SVM linéaire** (`LinearSVC`) pour prédire le genre du roman à partir du BoW des personnages.
* Tester plusieurs valeurs de l’hyperparamètre `C` (par ex. : 0.1, 1, 3, 10).
* Utiliser une **validation croisée à 5 plis** pour évaluer les performances.
* Comparer les scores : `accuracy`, `f1_macro`.

### 6. Évaluation finale

* Évaluer le modèle optimal sur le jeu de test.
* Afficher le **rapport de classification** (`classification_report`) et la **matrice de confusion**.
* Identifier et commenter **quelques erreurs de classification** :
  les personnages de certains genres sont-ils plus difficiles à classer ? pourquoi ?

## Bonus

* Visualiser les **10 mots les plus caractéristiques** de chaque genre selon les coefficients du SVM linéaire.

---

## Fichiers nécessaires

```
data/
 ├── characters/
 │     ├── roman1.book
 │     ├── roman2.book
 │     ├── ...
 └── genre_labels.json
```


In [1]:
import os
import re
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm

In [2]:
def read_booknlp(path_book):
    with open(path_book, "r", encoding="utf-8") as file:
        lines = file.readlines()
        # eval() transforme une chaîne de caractères représentant un dictionnaire en objet Python
        dicts = [eval(line.strip()) for line in lines if line.strip()]
    return dicts[:10]# On ne prends que les 10 premiers personnages pour un roman

In [6]:
def get_characterization(booknlp_data):
    list_tout_mots = []
    for i in range(len(booknlp_data)):
        list_mots_par_perso = []
        list_mots_par_perso.extend([item['w'] for item in booknlp_data[i]['agent']])
        list_mots_par_perso.extend([item['w'] for item in booknlp_data[i]['patient']])
        list_mots_par_perso.extend([item['w'] for item in booknlp_data[i]['mod']])
        list_tout_mots.append(list_mots_par_perso)
        
    return list_tout_mots

**Cliquez sur le lien suivant pour télécharger les données**  
[Télécharger le fichier](https://filesender.renater.fr/?s=download&token=16b800d7-842d-4f43-8578-72e007fc7970)

In [11]:
# Charger tous les fichiers .book depuis le dossier "BOOK_POLICIER_SENTIMENT"
books_policier_sentiment = glob.glob(os.path.join("BOOKS_POLICIER_SENTIMENT/", "*.book"))
#book_files = glob.glob(os.path.join("/home/crazyjeannot/OUTPUT_BOOK_GEN_Z/", "*.book"))

In [26]:
def get_bow_characters(top_N_words, book_files):
    bow_characters = {}  # Dictionary to store the BoW for each character
    
    for filepath in book_files:
        filename = os.path.basename(filepath)
        book_data = read_booknlp(filepath)  # Read the 5 first characters
        # Use the updated get_characterization to obtain a list of tokens per character
        characters_tokens = get_characterization(book_data)
        for idx, tokens in enumerate(characters_tokens):
            total = len(tokens)
            counter_tokens = Counter(tokens)
            bow = {}
            for word in top_N_words:
                bow[word] = counter_tokens.get(word, 0) / total if total > 0 else 0
            
            # Try to get the character's name from the 'mentions' field; otherwise, use a default name.
            if "mentions" in book_data[idx] and book_data[idx]["mentions"].get("proper") and len(book_data[idx]["mentions"]["proper"]) > 0:
                char_name = book_data[idx]["mentions"]["proper"][0]["n"].lower()
            else:
                char_name = f"char{idx+1}"
            
            key = f"{filename} --- {char_name}"
            bow_characters[key] = bow

    df_bow_characters = pd.DataFrame.from_dict(bow_characters, orient="index", columns=top_N_words)
    df_bow_characters.to_csv("BoW_personnages.csv", index=True)        
    return df_bow_characters

In [27]:
import pickle
with open('LISTE_1000_MOTS.pkl', 'rb') as file:
    top_1000_words = pickle.load(file)

In [28]:
df_personnages_POLICIER_SENTIMENT = get_bow_characters(top_1000_words, books_policier_sentiment)

In [22]:
df_personnages_POLICIER_SENTIMENT = pd.read_csv("BoW_personnages.csv", index_col="Unnamed: 0")

In [23]:
df_personnages_POLICIER_SENTIMENT

,avoir,dire,voir,faire,vouloir,pouvoir,savoir,aller,aimer,regarder,...,supprimer,avis,adosser,médecin,défaire,décourager,armer,délaisser,déplaire,creuser
1931_Simenon-Georges_La-Guinguette-a-deux-sous.book --- james,0.069343,0.023723,0.016423,0.031022,0.025547,0.014599,0.023723,0.023723,0.012774,0.041971,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1931_Simenon-Georges_La-Guinguette-a-deux-sous.book --- maigret,0.059730,0.032755,0.038536,0.009634,0.017341,0.017341,0.017341,0.011561,0.003854,0.040462,...,0.000000,0.003854,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1931_Simenon-Georges_La-Guinguette-a-deux-sous.book --- basso,0.043257,0.012723,0.020356,0.012723,0.020356,0.010178,0.017812,0.025445,0.015267,0.025445,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1931_Simenon-Georges_La-Guinguette-a-deux-sous.book --- victor,0.059259,0.037037,0.022222,0.022222,0.014815,0.033333,0.037037,0.011111,0.000000,0.025926,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1931_Simenon-Georges_La-Guinguette-a-deux-sous.book --- mme basso,0.048980,0.036735,0.032653,0.036735,0.012245,0.012245,0.012245,0.028571,0.012245,0.020408,...,0.004082,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1928_Delly_Cœurs-ennemis-Laquelle.book --- humphrey,0.079137,0.053957,0.025180,0.028777,0.017986,0.007194,0.032374,0.007194,0.010791,0.010791,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1928_Delly_Cœurs-ennemis-Laquelle.book --- faustina,0.045455,0.025974,0.025974,0.032468,0.025974,0.019481,0.000000,0.012987,0.025974,0.032468,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1928_Delly_Cœurs-ennemis-Laquelle.book --- orietta et faustina,0.085938,0.007812,0.046875,0.015625,0.007812,0.015625,0.007812,0.007812,0.015625,0.015625,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1928_Delly_Cœurs-ennemis-Laquelle.book --- my lady,0.051020,0.020408,0.040816,0.040816,0.040816,0.000000,0.000000,0.020408,0.000000,0.010204,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
def add_genre_column(df_bow, path_labels="genre_labels.json"):
    # Charger le fichier JSON
    with open(path_labels, "r", encoding="utf-8") as f:
        labels = json.load(f)
    
    # Construire un dictionnaire {nom du roman .txt -> genre}
    title_to_genre = {
        item["title"].replace(".txt", ""): item["genre"]
        for item in labels["texts"]
    }
    
    # Extraire le nom du roman à partir de l’index du DataFrame
    def extract_book_name(row_index):
        # Ex : '1931_Simenon-Georges_La-Guinguette-a-deux-sous.book --- james'
        base = row_index.split(" --- ")[0]
        return base.replace(".book", "")  # correspond à la clé dans le JSON sans .txt
    
    # Associer chaque ligne à son genre
    df_bow = df_bow.copy()
    df_bow["genre"] = [
        title_to_genre.get(extract_book_name(idx), "Inconnu")
        for idx in df_bow.index
    ]
    
    return df_bow

In [30]:
df_personnages_POLICIER_SENTIMENT_genred = add_genre_column(df_personnages_POLICIER_SENTIMENT)

In [31]:
df_personnages_POLICIER_SENTIMENT_genred

,avoir,dire,voir,faire,vouloir,pouvoir,savoir,aller,aimer,regarder,...,avis,adosser,médecin,défaire,décourager,armer,délaisser,déplaire,creuser,genre
1931_Simenon-Georges_La-Guinguette-a-deux-sous.book --- james,0.069343,0.023723,0.016423,0.031022,0.025547,0.014599,0.023723,0.023723,0.012774,0.041971,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,policier
1931_Simenon-Georges_La-Guinguette-a-deux-sous.book --- maigret,0.059730,0.032755,0.038536,0.009634,0.017341,0.017341,0.017341,0.011561,0.003854,0.040462,...,0.003854,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,policier
1931_Simenon-Georges_La-Guinguette-a-deux-sous.book --- basso,0.043257,0.012723,0.020356,0.012723,0.020356,0.010178,0.017812,0.025445,0.015267,0.025445,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,policier
1931_Simenon-Georges_La-Guinguette-a-deux-sous.book --- victor,0.059259,0.037037,0.022222,0.022222,0.014815,0.033333,0.037037,0.011111,0.000000,0.025926,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,policier
1931_Simenon-Georges_La-Guinguette-a-deux-sous.book --- mme basso,0.048980,0.036735,0.032653,0.036735,0.012245,0.012245,0.012245,0.028571,0.012245,0.020408,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,policier
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1928_Delly_Cœurs-ennemis-Laquelle.book --- humphrey,0.079137,0.053957,0.025180,0.028777,0.017986,0.007194,0.032374,0.007194,0.010791,0.010791,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sentimental
1928_Delly_Cœurs-ennemis-Laquelle.book --- faustina,0.045455,0.025974,0.025974,0.032468,0.025974,0.019481,0.000000,0.012987,0.025974,0.032468,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sentimental
1928_Delly_Cœurs-ennemis-Laquelle.book --- orietta et faustina,0.085938,0.007812,0.046875,0.015625,0.007812,0.015625,0.007812,0.007812,0.015625,0.015625,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sentimental
1928_Delly_Cœurs-ennemis-Laquelle.book --- my lady,0.051020,0.020408,0.040816,0.040816,0.040816,0.000000,0.000000,0.020408,0.000000,0.010204,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sentimental


In [35]:
df_personnages_POLICIER_SENTIMENT_genred.to_csv("BoW_personnages_Genres.csv", index=True)